<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #0284c7; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        LazyFrame, Optimizador de Consultas y Streaming 🧠🌊
      </h1>
      <p style="margin: 6px 0 0 0; color: #0284c7; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Polars de Alto Rendimiento
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #0284c7; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 11 Extra
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #0284c7; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/11%20-%20Polars/03_LazyFrame_Optimizador_de_Consultas_y_Streaming.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. El Modo Perezoso (*Lazy Evaluation*) y el `LazyFrame` 🛋️

En modo imperativo tradicional (*Eager*), cada línea de código procesa datos de inmediato en memoria. En modo perezoso (*Lazy*), Polars **no ejecuta nada inmediatamente**: construye un **Grafo Acíclico Dirigido (DAG)** que describe la intención del usuario.

### ¿Por qué Lazy es infinitamente superior para Big Data?
* **Predicate Pushdown:** Aplica filtros en el origen (ej. leyendo solo registros del 2025 del disco).
* **Projection Pushdown:** Lee únicamente las columnas necesarias, ignorando las demás en disco.
* **Slice Pushdown:** Si pides `.head(10)`, detiene la lectura en cuanto obtiene 10 filas.

In [58]:
import polars as pl
import numpy as np
import os

import os, urllib.request, urllib.parse

def load_dataset(filename, module_name="11 - Polars"):
    """
    Carga o descarga de forma segura el dataset para ejecución local o en Google Colab.
    Si no se encuentra localmente ni en GitHub, lo genera automáticamente.
    """
    candidates = [
        os.path.join("data", filename),
        os.path.join(module_name, "data", filename),
        os.path.join("..", "data", filename),
        os.path.join("..", module_name, "data", filename),
        os.path.join("Data Science programming", module_name, "data", filename),
        os.path.join("..", "Data Science programming", module_name, "data", filename),
        filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
            
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    folder_path = f"Data Science programming/{module_name}"
    encoded_folder = urllib.parse.quote(folder_path)
    encoded_file = urllib.parse.quote(filename)
    
    urls = [
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{encoded_folder}/data/{encoded_file}",
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/master/{encoded_folder}/data/{encoded_file}"
    ]
    
    print(f"📥 Intentando descargar '{filename}' desde el repositorio oficial...")
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    with open(target_path, 'wb') as out_f:
                        out_f.write(response.read())
                    print(f"✅ Dataset '{filename}' descargado exitosamente.")
                    return target_path
        except Exception:
            continue
            
    print(f"⚙️ Generando '{filename}' sintéticamente para ejecución inmediata...")
    import polars as pl
    import numpy as np
    np.random.seed(42)
    
    n_clientes = 1000
    df_c = pl.DataFrame({
        'id_cliente': [f'CLI-{i:04d}' for i in range(1, n_clientes + 1)],
        'nombre': [f'Cliente_{i}' for i in range(1, n_clientes + 1)],
        'segmento': np.random.choice(['Corporativo', 'Pyme', 'Consumo', 'Gobierno'], n_clientes),
        'edad': np.random.randint(18, 70, n_clientes),
        'ciudad_residencia': np.random.choice(['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Bucaramanga'], n_clientes),
        'ingreso_anual': np.random.normal(45000000, 15000000, n_clientes).round(2)
    })
    
    n_ventas = 60000
    cats = ['Tecnología', 'Mobiliario', 'Material de Oficina', 'Servicios']
    prods = ['Laptop Pro', 'Monitor 4K', 'Silla Ergonómica', 'Escritorio', 'Papel A4', 'Tóner', 'Mantenimiento']
    ciudades = ['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Barranquilla']
    
    cant = np.random.randint(1, 10, n_ventas)
    pu = np.random.choice([25000.0, 120000.0, 450000.0, 1200000.0, 3500000.0], n_ventas)
    desc = np.random.choice([0.0, 0.05, 0.10, 0.15], n_ventas)
    tot = (cant * pu * (1 - desc)).round(2)
    
    df_v = pl.DataFrame({
        'id_venta': [f'VNT-{i:06d}' for i in range(1, n_ventas + 1)],
        'fecha': [f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}' for _ in range(n_ventas)],
        'id_cliente': np.random.choice(df_c['id_cliente'], n_ventas),
        'categoria': np.random.choice(cats, n_ventas),
        'producto': np.random.choice(prods, n_ventas),
        'cantidad': cant,
        'precio_unitario': pu,
        'descuento': desc,
        'ciudad_venta': np.random.choice(ciudades, n_ventas),
        'total_venta': tot
    })
    
    c_csv_path = os.path.join("data", "clientes.csv")
    c_pq_path = os.path.join("data", "clientes.parquet")
    v_csv_path = os.path.join("data", "ventas.csv")
    v_pq_path = os.path.join("data", "ventas.parquet")
    
    if not os.path.exists(c_csv_path): df_c.write_csv(c_csv_path)
    if not os.path.exists(c_pq_path): df_c.write_parquet(c_pq_path)
    if not os.path.exists(v_csv_path): df_v.write_csv(v_csv_path)
    if not os.path.exists(v_pq_path): df_v.write_parquet(v_pq_path)
    
    print(f"✅ Datasets preparados exitosamente en '{target_path}'.")
    return target_path

ruta_csv = load_dataset("ventas.csv")
lazy_df = pl.scan_csv(ruta_csv)
print(f"🚀 Polars versión: {pl.__version__}")
print(f"LazyFrame preparado: {type(lazy_df)}")

🚀 Polars versión: 1.35.2
LazyFrame preparado: <class 'polars.lazyframe.frame.LazyFrame'>


---
## 2. Conocer los Datos sin Cargarlos: `.collect_schema()` 🧬

Una de las señas de identidad de un `LazyFrame` es que **conoce la forma de los datos sin haberlos leído por completo**. Polars analiza el archivo (encabezados, tipos inferidos de una muestra) y construye un esquema que puedes consultar en microsegundos:

* `.collect_schema()`: devuelve un `Schema` (nombre de columna → dtype) sin ejecutar la consulta completa.
* `.collect_schema().names()`: solo los nombres de columnas.

Esto es clave para depurar pipelines de Big Data: puedes validar que las columnas y tipos son los esperados **antes** de lanzar una consulta pesada sobre terabytes de datos.

In [59]:
# El esquema se conoce sin materializar ni una sola fila de datos
esquema = lazy_df.collect_schema()
print("Esquema (columna -> dtype):")
print(esquema)
print("\nSolo los nombres:", esquema.names())

Esquema (columna -> dtype):
Schema({'id_venta': String, 'fecha': String, 'id_cliente': String, 'categoria': String, 'producto': String, 'cantidad': Int64, 'precio_unitario': Float64, 'descuento': Float64, 'ciudad_venta': String, 'total_venta': Float64})

Solo los nombres: ['id_venta', 'fecha', 'id_cliente', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento', 'ciudad_venta', 'total_venta']


---
## 3. Inspección del Plan de Optimización con `.explain()` 🔍

Podemos auditar exactamente cómo el motor en Rust optimizará nuestra consulta antes de gastar un solo ciclo de CPU:

In [60]:
consulta_lazy = (
    lazy_df
    .filter(pl.col("categoria") == "Tecnología")
    .select(["id_venta", "producto", "cantidad", "total_venta"])
    .filter(pl.col("total_venta") > 2000000)
)

print("PLAN LÓGICO OPTIMIZADO (Explain):")
print(consulta_lazy.explain())

PLAN LÓGICO OPTIMIZADO (Explain):
simple π 4/4 ["id_venta", "producto", ... 2 other columns]
  Csv SCAN [data/ventas.csv]
  PROJECT 5/10 COLUMNS
  SELECTION: [([(col("total_venta")) > (2e6)]) & ([(col("categoria")) == ("Tecnología")])]
  ESTIMATED ROWS: 59597


> 💡 **Cómo leer el plan (de abajo hacia arriba):**
> * `Csv SCAN [...]`: el punto de partida — Polars aún no ha tocado el disco al construir el plan.
> * `PROJECT 5/10 COLUMNS`: gracias al **projection pushdown**, el escaneo solo traerá 5 de las 10 columnas del CSV; las demás nunca se leen.
> * `SELECTION: ...`: gracias al **predicate pushdown**, ambos `.filter()` se combinaron en una sola condición booleana que se aplica **en el mismo paso de lectura**, no después.
> * `ESTIMATED ROWS`: una estimación (no el conteo real) que el optimizador usa para decidir estrategias, por ejemplo en joins.
>
> Fíjate en algo importante: en el código escribimos `.filter()` → `.select()` → `.filter()` (tres pasos separados), pero el plan optimizado los fusiona en un único `PROJECT` + `SELECTION` justo sobre el escaneo. **Nunca se materializa un DataFrame intermedio.**

---
## 4. El Optimizador No Lee tu Código en Orden: Reordena por Ti 🔀

Una prueba contundente de que Polars *piensa* en vez de solo *ejecutar línea por línea*: no importa en qué orden escribas `.filter()` y `.select()`, el optimizador produce **el mismo plan físico óptimo** (filtrar y proyectar lo antes posible, junto al escaneo).

In [61]:
# Orden A: filtrar primero, luego seleccionar columnas
consulta_a = (
    lazy_df
    .filter(pl.col("categoria") == "Tecnología")
    .select(["categoria", "total_venta", "producto"])
)

# Orden B: seleccionar primero, luego filtrar (en apariencia "menos eficiente")
consulta_b = (
    lazy_df
    .select(["categoria", "total_venta", "producto"])
    .filter(pl.col("categoria") == "Tecnología")
)

plan_a = consulta_a.explain()
plan_b = consulta_b.explain()

print("¿El plan optimizado es idéntico sin importar el orden escrito?", plan_a == plan_b)
print("\nPlan resultante (ambos casos):")
print(plan_a)

¿El plan optimizado es idéntico sin importar el orden escrito? True

Plan resultante (ambos casos):
simple π 3/3 ["categoria", "total_venta", ... 1 other column]
  Csv SCAN [data/ventas.csv]
  PROJECT 3/10 COLUMNS
  SELECTION: [(col("categoria")) == ("Tecnología")]
  ESTIMATED ROWS: 59597


---
## 5. Joins Perezosos: la Optimización También Atraviesa `.join()` 🔗

El pushdown no se detiene en un único `LazyFrame`: cuando unes (`.join()`) dos consultas perezosas, Polars empuja los filtros y las proyecciones **a cada lado del join por separado**, antes de combinarlos. Esto evita construir tablas intermedias gigantes.

In [62]:
ruta_clientes = load_dataset("clientes.csv")
lazy_clientes = pl.scan_csv(ruta_clientes)

consulta_join = (
    lazy_df
    .filter(pl.col("total_venta") > 1_000_000)
    .join(
        lazy_clientes.select(["id_cliente", "segmento", "ciudad_residencia"]),
        on="id_cliente",
        how="left",
    )
    .group_by("segmento")
    .agg(pl.col("total_venta").sum().alias("total_segmento"))
    .sort("total_segmento", descending=True)
)

print("PLAN DEL JOIN PEREZOSO:")
print(consulta_join.explain())

print("\nResultado materializado:")
print(consulta_join.collect())

PLAN DEL JOIN PEREZOSO:
SORT BY [descending: [true]] [col("total_segmento")]
  AGGREGATE[maintain_order: false]
    [col("total_venta").sum().alias("total_segmento")] BY [col("segmento")]
    FROM
    simple π 2/2 ["total_venta", "segmento"]
      LEFT JOIN:
      LEFT PLAN ON: [col("id_cliente")]
        Csv SCAN [data/ventas.csv]
        PROJECT 2/10 COLUMNS
        SELECTION: [(col("total_venta")) > (1e6)]
        ESTIMATED ROWS: 59597
      RIGHT PLAN ON: [col("id_cliente")]
        Csv SCAN [data/clientes.csv]
        PROJECT 2/6 COLUMNS
        ESTIMATED ROWS: 1006
      END LEFT JOIN

Resultado materializado:
shape: (4, 2)
┌─────────────┬────────────────┐
│ segmento    ┆ total_segmento │
│ ---         ┆ ---            │
│ str         ┆ f64            │
╞═════════════╪════════════════╡
│ Gobierno    ┆ 8.1352e10      │
│ Corporativo ┆ 7.4345e10      │
│ Pyme        ┆ 6.6284e10      │
│ Consumo     ┆ 6.5790e10      │
└─────────────┴────────────────┘


---
## 6. Materialización con `.collect()` y el Motor de Streaming 🌊

Para ejecutar el plan optimizado, invocamos `.collect()`. Por defecto usa el motor **in-memory** (`engine="auto"`), pero si el dataset no cabe cómodo en RAM —o simplemente queremos reducir la presión de memoria— podemos pedirle a Polars que use el **motor de streaming**, que procesa la consulta por lotes (*batches*) en vez de cargar todo de una vez:

```python
df = consulta_lazy.collect(engine="streaming")
```

> ⚠️ **Nota de versión:** en versiones antiguas de Polars (0.x) viste `collect(streaming=True)`. Ese parámetro quedó **deprecado**; en la serie 1.x actual el motor de ejecución se selecciona con `engine="streaming"` (también existen `"in-memory"`, el motor por defecto, y `"gpu"` si cuentas con una GPU NVIDIA compatible). Si el motor elegido no puede resolver la consulta, Polars cae de vuelta al motor in-memory de forma transparente.

In [63]:
# Ejecución con el motor in-memory (por defecto)
df_resultado = consulta_lazy.collect()
print(f"Filas resultantes tras optimización: {df_resultado.height}")
display(df_resultado.head(5))

# Ejecución con el motor de streaming (procesa por lotes, menor presión de RAM)
df_streaming = consulta_lazy.collect(engine="streaming")
print("Filas resultantes (streaming):", df_streaming.height)
print("¿Mismo resultado que el motor in-memory?", df_resultado.equals(df_streaming))

Filas resultantes tras optimización: 7278


id_venta,producto,cantidad,total_venta
str,str,i64,f64
"""VNT-000015""","""Monitor 4K""",5,5.7e6
"""VNT-000021""","""Mantenimiento""",6,6.48e6
"""VNT-000023""","""Papel A4""",6,6.48e6
"""VNT-000033""","""Tóner""",7,2.3275e7
"""VNT-000047""","""Monitor 4K""",1,3.325e6


Filas resultantes (streaming): 7278
¿Mismo resultado que el motor in-memory? True


---
## 7. Escritura en Streaming con `sink_parquet` / `sink_csv`: Nunca Materializar Todo 💾

`.collect()` siempre termina guardando el resultado completo en RAM como un `DataFrame`. Cuando el **resultado en sí** es demasiado grande para la memoria, la alternativa es no materializarlo nunca en Python: escribirlo directamente a disco en streaming con `.sink_parquet()` o `.sink_csv()`. Polars procesa y escribe la consulta por lotes, sin construir jamás el `DataFrame` completo en memoria.

In [64]:
import tempfile

ruta_salida = os.path.join(tempfile.gettempdir(), "ventas_tecnologia.parquet")

# La consulta se ejecuta en streaming y se escribe directo a disco:
# el DataFrame completo nunca existe en memoria de Python.
(
    lazy_df
    .filter(pl.col("categoria") == "Tecnología")
    .sink_parquet(ruta_salida)
)

print("Archivo escrito en streaming:", os.path.exists(ruta_salida))

# Podemos volver a escanearlo perezosamente para verificar, sin cargarlo entero
conteo = pl.scan_parquet(ruta_salida).select(pl.len()).collect()
print("Filas escritas:", conteo.item())

Archivo escrito en streaming: True
Filas escritas: 15089


---
## 8. ¿Cuándo Usar Lazy y Cuándo Eager? 🧭

| Escenario | Recomendación |
|---|---|
| Exploración interactiva rápida de un archivo pequeño | Eager (`pl.read_csv`) — ver resultados de inmediato |
| Pipeline de producción sobre archivos grandes (CSV/Parquet) | Lazy (`pl.scan_csv` / `pl.scan_parquet`) — deja que el optimizador decida |
| Vas a encadenar varias transformaciones antes de necesitar el resultado | Lazy — el optimizador fusiona y reordena todo el pipeline |
| El resultado final también es enorme y no cabe en RAM | Lazy + `.sink_parquet()` / `.sink_csv()` — nunca materializar |
| Necesitas depurar paso a paso viendo cada DataFrame intermedio | Eager, o `.collect()` parcial en cada paso durante el desarrollo |

> 💡 Una práctica común: **desarrollar y depurar en eager** sobre una muestra pequeña, y **convertir a lazy** (`scan_*` en vez de `read_*`) cuando el pipeline ya funciona y se va a correr sobre el dataset completo.

---
## 9. Ejercicio Práctico: Auditoría de un Pipeline Perezoso 🧪

Usando `lazy_df` (ventas) y `lazy_clientes` (clientes), construye —en un único `LazyFrame`, sin llamar `.collect()` hasta el final— una consulta que:

1. Filtre solo las ventas de la ciudad `"Bogotá"`.
2. Una (`.join()`) con `lazy_clientes` por `id_cliente`, trayendo únicamente las columnas `segmento` y `edad`.
3. Agrupe por `segmento` y calcule el **total vendido** (`.sum()`) y el **número de ventas** (`pl.len()`).
4. Ordene el resultado de mayor a menor total vendido.
5. Antes de `.collect()`, imprime `.explain()` y verifica que el filtro por ciudad aparece empujado junto al `SCAN` de ventas.

Escribe tu solución en la celda de abajo antes de revisar la respuesta guiada:

In [65]:
# 1-4. Construye la consulta perezosa (no llames .collect() todavía)
# consulta_bogota = (
#     lazy_df
#     ...
# )

# 5. Audita el plan
# print(consulta_bogota.explain())

# Materializa al final
# resultado = consulta_bogota.collect()
# display(resultado)

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
consulta_bogota = (
    lazy_df
    .filter(pl.col("ciudad_venta") == "Bogotá")
    .join(
        lazy_clientes.select(["id_cliente", "segmento", "edad"]),
        on="id_cliente",
        how="left",
    )
    .group_by("segmento")
    .agg(
        pl.col("total_venta").sum().alias("total_vendido"),
        pl.len().alias("n_ventas"),
    )
    .sort("total_vendido", descending=True)
)

print(consulta_bogota.explain())

resultado = consulta_bogota.collect()
display(resultado)
```
</details>

---
## 10. Resumen y Próximos Pasos 📌

| Concepto | Idea Clave |
|---|---|
| **`LazyFrame`** | Representa un plan de consulta (DAG), no datos cargados; se construye con `pl.scan_csv` / `pl.scan_parquet`. |
| **`.collect_schema()`** | Consulta nombres y tipos de columnas sin ejecutar la consulta. |
| **`.explain()`** | Audita el plan optimizado: qué columnas se proyectan, qué filtros se empujan al escaneo. |
| **Predicate / Projection Pushdown** | Los filtros y la selección de columnas se aplican lo antes posible, sin importar el orden en que los escribiste. |
| **Joins perezosos** | El pushdown atraviesa `.join()`: cada lado se filtra y proyecta por separado antes de combinarse. |
| **`.collect(engine=...)`** | `"in-memory"` (por defecto) materializa todo en RAM; `"streaming"` procesa por lotes con menor presión de memoria. |
| **`.sink_parquet()` / `.sink_csv()`** | Escritura en streaming a disco: el resultado completo nunca existe como `DataFrame` en Python. |

➡️ **Siguiente paso:** en el **Cuaderno 04 — Interoperabilidad y Benchmark: Pandas vs. Polars**, el cierre del módulo, mediremos con cronómetro en mano cuánto más rápido es Polars frente a Pandas, y veremos cómo intercambiar datos sin costo entre ambas librerías vía Apache Arrow.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Módulo Extra: Polars de Alto Rendimiento</i>
  </p>
</div>